# NIST TN 1822 — Verif.1.1: Pre-evacuation time distributions

Verify the model applies each pre-evacuation distribution correctly. Section 3.1.1; printed page 16 / PDF page 22.

Reference: <https://nvlpubs.nist.gov/nistpubs/technicalnotes/NIST.TN.1822.pdf>

In [1]:
from datetime import datetime
print(f"Executed on {datetime.now().astimezone().strftime('%d %B %Y, %H:%M %Z')}")

Executed on 23 May 2026, 09:37 UTC


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pedpy
from shapely.geometry import Point, Polygon

from jupedsim_scenarios import load_scenario, run_scenario

In [3]:
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f7f7f5",
    "axes.edgecolor": "#3a3a3a",
    "axes.labelcolor": "#1d1d1d",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.figsize": (8, 5),
})

In [4]:
from scenario_builders.nist1_1_premovement import build_variants, sample_reference

## Load the base scenario

In [5]:
SCENARIO_ZIP = Path("scenario_files") / "Nist-1-1-premovement.zip"
base = load_scenario(str(SCENARIO_ZIP))
print(base.summary())

Scenario: /work/standards/nist/scenario_files/Nist-1-1-premovement.zip
  Model:         CollisionFreeSpeedModel
  Seed:          42
  Max time:      2400s
  Exits:         1
  Distributions: 1
  Stages:        0
  Zones:         0
  Journeys:      1
  Agents:        ~10
  Journey elems: 2
  Route:         1 distribution, 0 checkpoint, 1 exit
  Sequence:      jps-distributions_0 -> jps-exits_0
    jps-distributions_0: 10 agents


## Sweep the four NIST distributions

`build_variants` deep-copies the base scenario, overrides `premovement_*` via `set_agent_params`, and bumps `max_time` to a value appropriate for the distribution's tail.

In [6]:
runs = {}
for case, variant in build_variants(base):
    result = run_scenario(variant, seed=42)
    df = result.trajectory_dataframe()
    # First non-stationary frame per agent = observed start time.
    starts = []
    for agent_id, sub in df.sort_values(['id', 'frame']).groupby('id'):
        x0, y0 = sub.iloc[0][['x', 'y']]
        moved = sub[(sub.x - x0).abs() + (sub.y - y0).abs() > 0.05]
        t = (moved.iloc[0]['frame'] / result.frame_rate) if len(moved) else float('nan')
        starts.append(t)
    runs[case.name] = {'case': case, 'observed_starts': np.array(starts)}
    result.cleanup()

Using fallback logic: No journeys defined
Processing with parameters: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'use_flow_spawning': False, 'use_premovement': True, 'premovement_distribution': 'uniform', 'premovement_param_a': 10.0, 'premovement_param_b': 100.0}
Using default parameters: v0=1.2, radius=0.15, n_agents=10

Distribution jps-distributions_0: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement': True, 'premovement_distribution': 'uniform', 'premovement_param_a': 10.0, 'premovement_param_b': 100.0, 'premovement_seed': None, 'radius_distribution': 'constant', 'radius_std': None, 'v0_distribution': 'constant', 'v0_std': None}
Generating premovement times: uniform with params {'a': 10.0, 'b': 100.0}, seed=1042
Premovement times stats - Min: 11.32s, M

Using fallback logic: No journeys defined
Processing with parameters: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'use_flow_spawning': False, 'use_premovement': True, 'premovement_distribution': 'gamma', 'premovement_param_a': 1.291, 'premovement_param_b': 103.901}
Using default parameters: v0=1.2, radius=0.15, n_agents=10

Distribution jps-distributions_0: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement': True, 'premovement_distribution': 'gamma', 'premovement_param_a': 1.291, 'premovement_param_b': 103.901, 'premovement_seed': None, 'radius_distribution': 'constant', 'radius_std': None, 'v0_distribution': 'constant', 'v0_std': None}
Generating premovement times: gamma with params {'a': 1.291, 'b': 103.901}, seed=1042
Premovement times stats - Min: 19.86s

Using fallback logic: No journeys defined
Processing with parameters: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'use_flow_spawning': False, 'use_premovement': True, 'premovement_distribution': 'lognormal', 'premovement_param_a': 4.586, 'premovement_param_b': 0.967}
Using default parameters: v0=1.2, radius=0.15, n_agents=10

Distribution jps-distributions_0: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement': True, 'premovement_distribution': 'lognormal', 'premovement_param_a': 4.586, 'premovement_param_b': 0.967, 'premovement_seed': None, 'radius_distribution': 'constant', 'radius_std': None, 'v0_distribution': 'constant', 'v0_std': None}
Generating premovement times: lognormal with params {'a': 4.586, 'b': 0.967}, seed=1042
Premovement times stats - Min: 

Using fallback logic: No journeys defined
Processing with parameters: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'radius_distribution': 'constant', 'v0_distribution': 'constant', 'use_flow_spawning': False, 'use_premovement': True, 'premovement_distribution': 'weibull', 'premovement_param_a': 139.285, 'premovement_param_b': 1.195}
Using default parameters: v0=1.2, radius=0.15, n_agents=10

Distribution jps-distributions_0: {'number': 10, 'radius': 0.15, 'v0': 1.2, 'distribution_mode': 'by_number', 'percentage': None, 'use_flow_spawning': False, 'flow_start_time': 0, 'flow_end_time': 10, 'use_premovement': True, 'premovement_distribution': 'weibull', 'premovement_param_a': 139.285, 'premovement_param_b': 1.195, 'premovement_seed': None, 'radius_distribution': 'constant', 'radius_std': None, 'v0_distribution': 'constant', 'v0_std': None}
Generating premovement times: weibull with params {'a': 139.285, 'b': 1.195}, seed=1042
Premovement times stats - Min: 

## Plot observed vs analytic

`sample_reference` re-uses the upstream distribution sampler so the overlay is by definition the same family the loader applied.

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, run) in zip(axes.flat, runs.items()):
    obs = run['observed_starts']
    obs = obs[~np.isnan(obs)]
    if len(obs):
        ax.hist(obs, bins=20, density=True, alpha=0.6, label='observed')
    ref = sample_reference(run['case'], 10000, seed=1)
    ax.hist(ref, bins=80, density=True, histtype='step', label='reference')
    ax.set_title(name)
    ax.set_xlabel('pre-evac time [s]')
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Acceptance

In [8]:
from scipy import stats as _stats
ALPHA = 0.05
rows = []
for name, run in runs.items():
    obs = run['observed_starts']
    obs = obs[~np.isnan(obs)]
    ref = sample_reference(run['case'], 10000, seed=2)
    ks = _stats.ks_2samp(obs, ref) if len(obs) else None
    rows.append({
        'case': name, 'n': int(len(obs)),
        'ks_p': ks.pvalue if ks else float('nan'),
    })
fit = pd.DataFrame(rows)
print(fit)
assert (fit['ks_p'] > ALPHA).all(), fit

        case   n      ks_p
0    uniform  10  0.273241
1      gamma  10  0.834412
2  lognormal  10  0.333898
3    weibull  10  0.997132
